# VOLO-D1 and Pretrained-ResNet Transfer Learning -- Google Colab

Standalone notebook for the two still-incomplete supplementary experiments:

| Experiment | Status |
|---|---|
| **VOLO-D1 face-only** | Stuck at epoch 1 -- a checkpoint exists (`best.pt`, restored automatically below) but Stage 1/Stage 2 never finished and it was never evaluated on the test split (no Table B row). This notebook resumes it to completion. |
| **Pretrained-ResNet18/ResNet50 bridge baseline** | Never run at all. `pretrained_resnet18` is the most interpretable comparison (same architecture family as the from-scratch Custom ResNet-18); `pretrained_resnet50` is optional. |

**Does not touch the core suite** (Experiments 0/0b/0c/A/B/C/D/E/F) -- it only
restores Experiment D's already-trained checkpoint from Drive as the reused
(never retrained) baseline row for Table B, exactly as
`scripts/run_transfer_learning.py` expects to find it
(`checkpoints/exp_d_shared_adapters_learned_balance_best_balanced_score.pt`
+ `outputs/metrics/exp_d_shared_adapters_learned_balance_test_metrics.json`).

Reuses the persistent-storage/resume machinery
(`src/training/persistent_artifacts.py::PersistentArtifactManager`) that
already restored the VOLO checkpoint's progress from Drive once before --
running this notebook again after any interruption is always safe.

## 1. User configuration

In [ ]:
# ============================================================
# USER CONFIGURATION -- edit this cell, then run the notebook top to bottom.
# ============================================================

REPO_URL = "https://github.com/adischwartz15/AgeGender.git"
REPO_BRANCH = "main"

# Reused only so this notebook's own logs/reports land alongside the core
# suite's in the same Drive folder -- change if your core run used a
# different RUN_ID. Does NOT need to already exist locally; this notebook
# creates it fresh if missing (unlike the core-suite notebook, nothing here
# depends on core-suite artifacts being restored into it).
PREVIOUS_RUN_ID = "2026-07-13_1210_core_seed42"

SEED = 42
USE_GOOGLE_DRIVE = True

# Reuse whichever dataset source your original Colab run used -- exactly
# one of these two must be set.
KAGGLE_DATASET_SLUG = None      # e.g. "jangedoo/utkface-new"
DRIVE_DATASET_DIR = None        # e.g. "/content/drive/MyDrive/AgeGender/data/utkface"

# --- Which experiments to run --------------------------------------------
RUN_VOLO = True
RUN_PRETRAINED_RESNET18 = True     # required for a real Table B bridge comparison
RUN_PRETRAINED_RESNET50 = False    # optional (capacity/depth comparison too)
TRANSFER_SEEDS = [42]  # only seed 42 is in scope; set e.g. [42, 123, 2026] to also run the others

print(f"RUN_VOLO={RUN_VOLO} RUN_PRETRAINED_RESNET18={RUN_PRETRAINED_RESNET18} "
      f"RUN_PRETRAINED_RESNET50={RUN_PRETRAINED_RESNET50} TRANSFER_SEEDS={TRANSFER_SEEDS}")


In [ ]:
# ============================================================
# Helper library -- shared plumbing used by every phase below.
# None of this reimplements model/dataset/training/evaluation logic; it only
# runs the repository's own scripts, and manages files/manifests around them.
# ============================================================

import datetime
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from IPython.display import Image, Markdown, display


def run_command(cmd, cwd=None, log_path=None, check=True, env=None):
    """Run a subprocess, streaming output live and optionally capturing it to a log file."""
    printable = cmd if isinstance(cmd, str) else " ".join(str(c) for c in cmd)
    print(f"$ {printable}")
    full_env = {**os.environ, **(env or {})}
    proc = subprocess.Popen(
        cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, shell=isinstance(cmd, str), env=full_env,
    )
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    proc.wait()
    if log_path:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text("".join(lines), encoding="utf-8")
    if check and proc.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {proc.returncode}): {printable}"
            + (f"\nSee log: {log_path}" if log_path else "")
        )
    return proc.returncode, "".join(lines)


def write_manifest(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(data, fh, indent=2, default=str)
    return path


def load_json(path):
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_copy2(src, dst):
    """shutil.copy2, but skips (with a message) instead of raising when src and dst resolve to the same file.

    This happens on a resumed run whose RUN_DIR already points directly at
    the persistent Drive/output run directory (e.g. a Colab run resumed
    from a Drive-backed RUN_DIR): the log/checkpoint/metrics files under
    RUN_DIR are already being written to their final destination, so a
    later "sync to persistent storage" copy step would otherwise raise
    shutil.SameFileError trying to copy a file onto itself.
    """
    src, dst = Path(src), Path(dst)
    if src.resolve() == dst.resolve():
        print(f"Skipping copy: source and destination are the same file ({dst})")
        return dst
    return shutil.copy2(src, dst)


def copy_tree_merge(src, dst):
    """Copy every file under src into dst (creating dst), without touching unrelated existing dst content."""
    src, dst = Path(src), Path(dst)
    if not src.exists():
        return []
    dst.mkdir(parents=True, exist_ok=True)
    copied = []
    for path in src.rglob("*"):
        if path.is_file():
            target = dst / path.relative_to(src)
            target.parent.mkdir(parents=True, exist_ok=True)
            safe_copy2(path, target)
            copied.append(target)
    return copied


def move_tree_clear(src, dst):
    """Move every file under src into dst, then remove the now-empty src.

    Used for the couple of scripts (generate_gradcam.py, run_robustness.py)
    whose output directory is not configurable per run, so consecutive
    experiments would otherwise collide.
    """
    copied = copy_tree_merge(src, dst)
    if Path(src).exists():
        shutil.rmtree(src)
    return copied


def flatten_overrides(obj, prefix=""):
    """Flatten a nested dict into a flat list of ["--set", "a.b.c=value", ...] CLI args.

    Mirrors src/utils/config.py's own --set parsing (parse_cli_overrides /
    _coerce_scalar) exactly, so every value round-trips the same way it
    would from a hand-typed CLI override.
    """
    args = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            new_prefix = f"{prefix}.{key}" if prefix else key
            args.extend(flatten_overrides(value, new_prefix))
    else:
        value = obj
        if value is None:
            value = "null"
        elif isinstance(value, bool):
            value = "true" if value else "false"
        args += ["--set", f"{prefix}={value}"]
    return args


def display_image(path, caption=None):
    path = Path(path)
    if not path.exists():
        print(f"(plot not available: {path})")
        return
    if caption:
        display(Markdown(f"**{caption}**"))
    display(Image(filename=str(path)))


def artifact_ready(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0


def validate_required_artifacts(paths):
    missing = [str(p) for p in paths if not artifact_ready(p)]
    if missing:
        raise RuntimeError("Required artifact(s) missing or empty:\n  " + "\n  ".join(missing))
    return True


print("Helper library loaded.")

## 2. Runtime, GPU, repository, and dependencies

In [ ]:
# ============================================================
# Detect Google Colab
# ============================================================

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in Google Colab: {IN_COLAB}")
if not IN_COLAB:
    print(
        "This notebook is designed for Google Colab. It may still run "
        "elsewhere, but paths under /content are Colab-specific; consider "
        "notebooks/train_evaluate_kaggle.ipynb or a local checkout instead."
    )

In [ ]:
# ============================================================
# Runtime and GPU verification
# ============================================================

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU memory (GB): {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")
    else:
        print(
            "No GPU detected -- training will run on CPU and will be substantially "
            "slower. Enable a GPU accelerator in this platform's runtime settings "
            "before running the training cells below."
        )
except ImportError:
    print("PyTorch not yet installed -- run the dependency installation cell below first.")

In [ ]:
# ============================================================
# Repository setup
# ============================================================

REPO_DIR = Path("/content/AgeGender")

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    print(f"Repository already present at {REPO_DIR}; fetching latest {REPO_BRANCH}...")
    run_command(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR)
    run_command(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR)
    run_command(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_DIR)
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_command(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")

In [ ]:
# ============================================================
# Dependency installation
# ============================================================

run_command(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=REPO_DIR,
)

# Editable install is a convenience for `import src...` directly from notebook
# cells; every script under scripts/ already inserts the repo root onto its
# own sys.path and does not depend on this succeeding. pyproject.toml also
# currently declares requires-python >=3.11, so this is skipped gracefully
# (not treated as fatal) on older interpreters.
try:
    run_command([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=REPO_DIR)
    print("Editable install succeeded.")
except Exception as exc:
    print(f"Editable install skipped ({exc}); continuing with sys.path only (this is expected and fine).")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import torch  # noqa: E402  (re-import after installation, in case this is a fresh runtime)

print("Dependencies installed.")

## 3. Google Drive and a lightweight run directory

Only `logs/`, `reports/`, and `manifests/` are needed here -- VOLO and the
pretrained-ResNet baselines manage their own persistent storage
independently (`AgeGender/transfer_learning/<model>/seed_<seed>/`, restored
automatically in section 6 below).

In [ ]:
# ============================================================
# Mount Google Drive (only if USE_GOOGLE_DRIVE=True)
# ============================================================

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted at /content/drive")
else:
    print("Google Drive not mounted (USE_GOOGLE_DRIVE=False or not running in Colab).")


def sync_after_phase(phase_label):
    """Copy the run directory to Drive after a major phase, when enabled."""
    if IN_COLAB and USE_GOOGLE_DRIVE:
        dest = Path("/content/drive/MyDrive/AgeGender/runs") / RUN_ID
        copy_tree_merge(RUN_DIR, dest)
        print(f"[drive sync: {phase_label}] -> {dest}")

In [ ]:
# ============================================================
# Lightweight run directory -- just enough for this notebook's own logs/
# reports/manifests, reusing PREVIOUS_RUN_ID so they land next to the core
# suite's in the same Drive folder.
# ============================================================

WORKSPACE_DIR = Path("/content/agegender_runs")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = PREVIOUS_RUN_ID
RUN_DIR = WORKSPACE_DIR / RUN_ID
for _sub in ("logs", "reports", "manifests"):
    (RUN_DIR / _sub).mkdir(parents=True, exist_ok=True)

print(f"RUN_ID  = {RUN_ID}")
print(f"RUN_DIR = {RUN_DIR}")


## 4. Restore Experiment D's checkpoint (the reused baseline)

`scripts/run_transfer_learning.py` never retrains the from-scratch
baseline -- for seed 42 specifically, it looks for exactly
`checkpoints/exp_d_shared_adapters_learned_balance_best_balanced_score.pt`
and (if present) reuses
`outputs/metrics/exp_d_shared_adapters_learned_balance_test_metrics.json`
instead of re-evaluating (see `collect_baseline_metrics` in that script).
Both are restored here from the core suite's Drive run folder.

In [ ]:
# ============================================================
# Restore Experiment D's checkpoint + test metrics from Drive into the
# exact default locations run_transfer_learning.py's baseline lookup reads.
# ============================================================

BASELINE_EXPERIMENT = "exp_d_shared_adapters_learned_balance"

if IN_COLAB and USE_GOOGLE_DRIVE:
    _drive_run_dir = Path("/content/drive/MyDrive/AgeGender/runs") / PREVIOUS_RUN_ID
    _baseline_checkpoint_src = _drive_run_dir / "checkpoints" / f"{BASELINE_EXPERIMENT}_best_balanced_score.pt"
    _baseline_metrics_src = _drive_run_dir / "metrics" / f"{BASELINE_EXPERIMENT}_test_metrics.json"

    (REPO_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
    (REPO_DIR / "outputs" / "metrics").mkdir(parents=True, exist_ok=True)

    if _baseline_checkpoint_src.exists():
        safe_copy2(_baseline_checkpoint_src, REPO_DIR / "checkpoints" / _baseline_checkpoint_src.name)
        print(f"Restored baseline checkpoint: {_baseline_checkpoint_src.name}")
    else:
        raise FileNotFoundError(
            f"No baseline checkpoint found at {_baseline_checkpoint_src} -- check PREVIOUS_RUN_ID above, "
            "or that the core-suite notebook actually completed Experiment D."
        )

    if _baseline_metrics_src.exists():
        safe_copy2(_baseline_metrics_src, REPO_DIR / "outputs" / "metrics" / _baseline_metrics_src.name)
        print(f"Restored baseline test metrics: {_baseline_metrics_src.name} (will be reused, not re-evaluated)")
    else:
        print(f"No pre-computed test metrics at {_baseline_metrics_src} -- will evaluate the checkpoint once instead.")
else:
    raise RuntimeError(
        "USE_GOOGLE_DRIVE=False or not in Colab -- cannot restore Experiment D's checkpoint. "
        "This notebook has no other way to obtain the from-scratch baseline."
    )


## 5. Dataset and locked split

Prepared at the plain **default** location (`data/splits/full_metadata_with_splits.csv`)
-- no path override needed here, since that is exactly where
`run_transfer_learning.py` (and VOLO's/pretrained-ResNet's own config
loaders) look for it by default.

In [ ]:
# ============================================================
# Kaggle credentials (only needed if KAGGLE_DATASET_SLUG is set below)
# Never printed, logged, or written to a manifest.
# ============================================================

if KAGGLE_DATASET_SLUG and not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        print("Loaded Kaggle credentials from Colab Secrets.")
    except Exception:
        import getpass
        print("Enter Kaggle API credentials (hidden input; never printed or logged):")
        os.environ["KAGGLE_USERNAME"] = getpass.getpass("KAGGLE_USERNAME: ")
        os.environ["KAGGLE_KEY"] = getpass.getpass("KAGGLE_KEY: ")
elif KAGGLE_DATASET_SLUG:
    print("Kaggle credentials already present in the environment.")
else:
    print("KAGGLE_DATASET_SLUG not set; skipping (DRIVE_DATASET_DIR will be used instead).")

In [ ]:
# ============================================================
# Dataset setup
# ============================================================

if DRIVE_DATASET_DIR:
    dest = REPO_DIR / "data" / "raw"
    dest.mkdir(parents=True, exist_ok=True)
    print(f"Using dataset already available in Drive: {DRIVE_DATASET_DIR}")
    copy_tree_merge(DRIVE_DATASET_DIR, dest)
elif KAGGLE_DATASET_SLUG:
    os.environ["KAGGLE_DATASET_SLUG"] = KAGGLE_DATASET_SLUG
    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        raise RuntimeError(
            "KAGGLE_USERNAME/KAGGLE_KEY are not set. Run the credentials cell "
            "above (Colab Secrets or hidden prompt) before this cell."
        )
    run_command(
        [sys.executable, "-u", "scripts/download_kaggle_data.py"],
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "download_data.log",
    )
else:
    raise RuntimeError(
        "No dataset source configured. Set KAGGLE_DATASET_SLUG (with Kaggle "
        "credentials available) or DRIVE_DATASET_DIR in the configuration "
        "cell before running this notebook."
    )

sync_after_phase("dataset_setup")

In [ ]:
# ============================================================
# Data validation and deterministic split preparation -- default paths,
# no RUN_DIR override (unlike the core-suite notebook), since this is
# exactly where run_transfer_learning.py's config loader looks.
# ============================================================

split_csv_path = REPO_DIR / "data" / "splits" / "full_metadata_with_splits.csv"

if split_csv_path.exists():
    print(f"Reusing existing prepared split at {split_csv_path}.")
else:
    run_command(
        [sys.executable, "-u", "scripts/prepare_data.py"],
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "prepare_data.log",
    )

if not split_csv_path.exists():
    raise FileNotFoundError(f"Split preparation did not produce {split_csv_path} -- check the log above.")

print(f"Split ready: {split_csv_path}")
sync_after_phase("data_preparation")


## 6. VOLO-D1 face-only transfer learning

Resumes exactly where the earlier partial run left off -- the bootstrap
cell below restores the epoch-1 checkpoint from Drive automatically, and
the authoritative Table B cell continues training (Stage 1 -> Stage 2)
and then evaluates on the test split, something the earlier run never
reached.

In [ ]:
# ============================================================
# VOLO dependency install and toggle.
# ============================================================

RUN_TRANSFER_LEARNING_EXTENSION = RUN_VOLO

if RUN_TRANSFER_LEARNING_EXTENSION:
    run_command(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-transfer.txt"],
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "transfer_learning_install.log",
    )
    print("timm/torchvision installed for the transfer-learning extensions.")
else:
    print("RUN_VOLO=False -- skipping VOLO entirely.")


### Persistent Transfer-Learning Storage and Resume

Everything below is **idempotent and safe to rerun after a runtime restart**
-- it does not depend on any Python variable from a previous, now-dead
kernel. After a disconnect, the only cells that need to be rerun are:
environment setup, repository checkout, dependency installation, this
bootstrap cell (which mounts Drive and restores whatever was already
synced), and the run cell further down.

**Design: fast local working directory + persistent Drive mirror.**
Writing every training artifact directly to Drive on every batch would
slow training down; instead, checkpoints are written to a fast local
directory first (`LOCAL_TRANSFER_ROOT`, under `/content`, which vanishes on
disconnect) and mirrored to Drive (`PERSISTENT_TRANSFER_ROOT`) after every
epoch (atomically, with checksums -- see
`src/training/persistent_artifacts.py::PersistentArtifactManager`). At
startup, this cell restores the newest valid checkpoint from Drive back
into the local directory before any training happens, so a fresh runtime
picks up exactly where the last one left off.

**Seed completion.** A seed only counts as complete once its
`completion.json` marker, best checkpoint, checksum, and test metrics all
validate against the current data split and model config -- never just
because a directory happens to exist (see
`PersistentArtifactManager.is_seed_complete`).


In [ ]:
# ============================================================
# Persistent Transfer-Learning Storage and Resume -- bootstrap cell.
# Self-contained: reconstructs every path/config this section needs, so
# rerunning environment setup + repository checkout + dependency
# installation + this cell is enough to resume a disconnected run. Safe to
# rerun any number of times.
# ============================================================

PLATFORM = "colab"
# TRANSFER_SEEDS is set in the user-configuration cell above (default [42])
AUTO_RESUME = True
SKIP_COMPLETED = True
SYNC_AFTER_EPOCH = True
SINGLE_SEED = None  # set e.g. 123 to use the optional single-seed cell further down instead of the full run

LOCAL_TRANSFER_ROOT = Path("/content/AgeGender_runtime/transfer_learning")
PERSISTENT_TRANSFER_ROOT = (
    Path("/content/drive/MyDrive/AgeGender/transfer_learning") if (IN_COLAB and USE_GOOGLE_DRIVE) else None
)
LOCAL_TRANSFER_ROOT.mkdir(parents=True, exist_ok=True)

if RUN_TRANSFER_LEARNING_EXTENSION:
    if str(REPO_DIR / "scripts") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "scripts"))
    from src.training.persistent_artifacts import PersistentArtifactManager, format_status_line, seed_status_report
    from src.utils.io import file_sha256

    _tl_split_path = REPO_DIR / "data" / "splits" / "full_metadata_with_splits.csv"
    _tl_split_sha256 = file_sha256(_tl_split_path) if _tl_split_path.exists() else None

    print(f"Local working root:    {LOCAL_TRANSFER_ROOT}")
    print(f"Persistent Drive root: {PERSISTENT_TRANSFER_ROOT or '(disabled -- USE_GOOGLE_DRIVE=False or not in Colab)'}")
    print(f"AUTO_RESUME={AUTO_RESUME}  SKIP_COMPLETED={SKIP_COMPLETED}  SYNC_AFTER_EPOCH={SYNC_AFTER_EPOCH}\n")

    if PERSISTENT_TRANSFER_ROOT is not None:
        for _seed in TRANSFER_SEEDS:
            _mgr = PersistentArtifactManager(
                "volo_d1_face_only_pretrained", _seed, LOCAL_TRANSFER_ROOT, PERSISTENT_TRANSFER_ROOT,
                sync_after_epoch=SYNC_AFTER_EPOCH,
            )
            _restored = _mgr.restore_seed()
            if _restored:
                print(f"Restored {len(_restored)} file(s) for seed {_seed} from Drive.")

    print("\nSeed status:")
    for _seed in TRANSFER_SEEDS:
        _status = seed_status_report(
            "volo_d1_face_only_pretrained", _seed, LOCAL_TRANSFER_ROOT, PERSISTENT_TRANSFER_ROOT,
            expected_split_sha256=_tl_split_sha256,
        )
        print("  " + format_status_line(_status))
else:
    print("Skipped (RUN_TRANSFER_LEARNING_EXTENSION=False).")


### Live Status Table (All Experiments / Seeds)

Reusable, read-only status scan -- reruns instantly at any point (before,
during, or after training) to show every experiment/seed found under
`LOCAL_TRANSFER_ROOT`: status (`COMPLETE`/`INCOMPLETE`/`NOT STARTED`),
current stage, epoch, best score, when it was last updated, and its
checkpoint path. Reads only the small `state/*.json` files
`PersistentArtifactManager` already writes after every epoch
(`src/training/persistent_artifacts.py::scan_artifact_root`) -- never loads
a full checkpoint, so it stays fast even with many seeds/experiments.


In [ ]:
# ============================================================
# Live status table -- safe to re-run any time, including while a training
# cell above is still running in another notebook session/kernel sharing
# the same LOCAL_TRANSFER_ROOT (e.g. to check progress from a second tab).
# ============================================================

from src.training.persistent_artifacts import scan_artifact_root


def show_artifact_status_table(root=None):
    import pandas as pd

    root = root if root is not None else LOCAL_TRANSFER_ROOT
    rows = scan_artifact_root(root)
    if not rows:
        print(f"No experiment/seed artifacts found yet under {root}.")
        return None
    table = pd.DataFrame(rows, columns=["experiment", "seed", "status", "stage", "epoch", "best_score", "last_update", "checkpoint"])
    table = table.sort_values(["experiment", "seed"]).reset_index(drop=True)
    display(table)
    return table


_ = show_artifact_status_table()


In [ ]:
# ============================================================
# Table B: multi-seed run (the authoritative source for the reported
# numbers) + comparison against the best from-scratch model
# (exp_d_shared_adapters_learned_balance, reused, never retrained here).
#
# --resume --skip-completed --sync-after-epoch: an already-complete seed is
# reused (not retrained); an interrupted seed resumes from its latest valid
# checkpoint; a seed that never started trains from scratch. See
# docs/transfer_learning.md "Colab recovery" for the manual recovery steps
# after a runtime disconnect (they are exactly: rerun setup, mount Drive,
# rerun this cell -- nothing else).
# ============================================================

if RUN_TRANSFER_LEARNING_EXTENSION:
    tl_seeds_arg = ",".join(str(s) for s in TRANSFER_SEEDS)
    tl_cmd = [
        sys.executable, "-u", str(REPO_DIR / "scripts" / "run_transfer_learning.py"),
        "--seeds", tl_seeds_arg,
        "--storage-root", str(LOCAL_TRANSFER_ROOT),
    ]
    if AUTO_RESUME:
        tl_cmd.append("--resume")
    if SKIP_COMPLETED:
        tl_cmd.append("--skip-completed")
    if SYNC_AFTER_EPOCH:
        tl_cmd.append("--sync-after-epoch")
    if PERSISTENT_TRANSFER_ROOT is not None:
        tl_cmd += ["--persistent-root", str(PERSISTENT_TRANSFER_ROOT)]

    run_command(tl_cmd, cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "transfer_learning_table_b.log")

    tl_table_b_path = REPO_DIR / "results" / "transfer_learning" / "table_b.csv"
    tl_manifest_path = REPO_DIR / "results" / "transfer_learning" / "table_b_manifest.json"
    if tl_table_b_path.exists():
        import pandas as pd
        tl_table_b = pd.read_csv(tl_table_b_path)
        display(tl_table_b)
        tl_manifest = load_json(tl_manifest_path) if tl_manifest_path.exists() else {}
        print(json.dumps(tl_manifest, indent=2))
        shutil.copy2(tl_table_b_path, RUN_DIR / "reports" / "table_b.csv")

        tl_summary_archive = LOCAL_TRANSFER_ROOT / "volo_d1_face_only_pretrained" / "transfer_learning_summary.zip"
        if tl_summary_archive.exists():
            safe_copy2(tl_summary_archive, RUN_DIR / "archives" / tl_summary_archive.name) \
                if (RUN_DIR / "archives").exists() else copy_tree_merge(tl_summary_archive.parent, RUN_DIR / "archives")
            print(f"Lightweight summary archive: {tl_summary_archive}")
            if PERSISTENT_TRANSFER_ROOT is not None:
                print(f"Mirrored to: {PERSISTENT_TRANSFER_ROOT / 'volo_d1_face_only_pretrained' / tl_summary_archive.name}")

        print("\nSeed status (after this run):")
        for _seed in TRANSFER_SEEDS:
            _status = seed_status_report(
                "volo_d1_face_only_pretrained", _seed, LOCAL_TRANSFER_ROOT, PERSISTENT_TRANSFER_ROOT,
                expected_split_sha256=_tl_split_sha256,
            )
            print("  " + format_status_line(_status))
    else:
        print(
            "Table B was not produced -- most likely no prepared split or no "
            "'exp_d_shared_adapters_learned_balance' checkpoint was available to reuse. "
            "See the log above for details."
        )
    sync_after_phase("transfer_learning")
else:
    print("Skipped (RUN_TRANSFER_LEARNING_EXTENSION=False).")


In [ ]:
# ============================================================
# Explicit scientific interpretation
# ============================================================

def _parse_leading_float(value):
    """build_transfer_learning_table renders seed-sensitive metrics as
    formatted strings, e.g. '5.234 (n=1, no std)' or '5.23 +/- 0.12 (n=3)' --
    extract the leading numeric value for comparison, or None if unparseable."""
    import math
    import re

    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    match = re.match(r"[-+]?\d*\.?\d+", str(value))
    return float(match.group()) if match else None


if RUN_TRANSFER_LEARNING_EXTENSION and tl_table_b_path.exists() and len(tl_table_b) == 2:
    _volo_row = tl_table_b[tl_table_b["Model"] == "volo_d1_face_only_pretrained"].iloc[0]
    _baseline_row = tl_table_b[tl_table_b["Model"] != "volo_d1_face_only_pretrained"].iloc[0]
    _single_seed = bool(tl_manifest.get("single_seed_no_variance_estimate", True))
    print(
        f"The externally pretrained VOLO-D1 system's Age MAE was '{_volo_row['Age MAE']}' "
        f"vs. the best from-scratch system's '{_baseline_row['Age MAE']}'; "
        f"Gender F1 was '{_volo_row['Gender F1']}' vs. '{_baseline_row['Gender F1']}'.\n\n"
        "This cannot be attributed to architecture (Transformer-style vs. residual CNN) "
        "alone, because initialization, pretraining data, capacity, input resolution, "
        "and optimizer/training schedule all differ simultaneously between the two rows -- "
        "see the explanation at the top of this section. "
        + ("Table B above is a SINGLE-SEED result with no variance estimate; treat any "
           "gap between the two rows as anecdotal, not a statistically supported claim." if _single_seed else
           "Table B above aggregates >=2 seeds per row (see the +/- std columns).")
    )
    _volo_mae, _baseline_mae = _parse_leading_float(_volo_row["Age MAE"]), _parse_leading_float(_baseline_row["Age MAE"])
    _volo_f1, _baseline_f1 = _parse_leading_float(_volo_row["Gender F1"]), _parse_leading_float(_baseline_row["Gender F1"])
    _volo_worse_on_age = _volo_mae is not None and _baseline_mae is not None and _volo_mae > _baseline_mae
    _volo_worse_on_gender = _volo_f1 is not None and _baseline_f1 is not None and _volo_f1 < _baseline_f1
    if _volo_worse_on_age or _volo_worse_on_gender:
        print(
            "\nOn at least one metric, the pretrained VOLO-D1 system performed WORSE than "
            "the from-scratch baseline. This is reported as-is -- a large pretrained model "
            "can overfit a dataset of only a few thousand images, and no hyperparameter was "
            "tuned against the test set to rescue this result."
        )
else:
    print(
        "Skipped or incomplete: RUN_TRANSFER_LEARNING_EXTENSION=False, or Table B does not "
        "have both rows (baseline checkpoint and/or VOLO run unavailable). No interpretation "
        "is printed without real numbers to interpret -- see the Table B cell above for why."
    )


### Pretrained ResNet-18 / ResNet-50 Bridge Baseline (Optional)

Reuses `scripts/run_transfer_learning.py` -- the same script, `TransferTrainer`,
and persistence layer as the VOLO-D1 section above -- with
`--model-family pretrained_resnet18` (the **required** pretraining bridge
baseline: the most interpretable comparison between this project's
from-scratch Custom ResNet-18 and a fully pretrained modern backbone,
since both share the ResNet architecture family) or `pretrained_resnet50`
(**optional**: a combined pretraining + architecture + capacity
comparison, never described as isolating pretraining alone). See
`src/models/pretrained_resnet.py`'s module docstring for the exact
scientific role and the documented implementation differences from this
project's own Custom ResNet-18 (torchvision's ResNet-18 is **not**
byte-identical to it).

Reuses the exact same seeds, locked split, adapters, heads, learned-
uncertainty loss balancing, and 2-stage transfer-training schedule as the
VOLO-D1 section -- only the backbone and its own official torchvision
preprocessing differ.

OFF by default. Set `RUN_PRETRAINED_RESNET_EXTENSION = True` to include it.


In [ ]:
# ============================================================
# Pretrained ResNet-18 (required) / ResNet-50 (optional) bridge baseline --
# reuses scripts/run_transfer_learning.py --model-family pretrained_resnet18
# (or pretrained_resnet50), i.e. the SAME TransferTrainer/persistence/
# Table-B machinery as the VOLO-D1 section above, just a different model
# family. Rebuilds results/transfer_learning/table_b.csv to include this
# row alongside the baseline and (if already run) VOLO-D1.
# ============================================================

if RUN_PRETRAINED_RESNET18:
    PRETRAINED_RESNET_FAMILY = "pretrained_resnet18"
    RUN_PRETRAINED_RESNET_EXTENSION = True
elif RUN_PRETRAINED_RESNET50:
    PRETRAINED_RESNET_FAMILY = "pretrained_resnet50"
    RUN_PRETRAINED_RESNET_EXTENSION = True
else:
    RUN_PRETRAINED_RESNET_EXTENSION = False

if RUN_PRETRAINED_RESNET_EXTENSION:
    if RUN_TRANSFER_LEARNING_EXTENSION:
        # timm is VOLO-only; the pretrained-ResNet extension needs
        # torchvision instead (also in requirements-transfer.txt) -- both
        # are installed together by the same requirements-transfer.txt, so
        # if the VOLO cell above already ran this install step, skip it here.
        pass
    else:
        run_command(
            [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-transfer.txt"],
            cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "pretrained_resnet_install.log",
        )

    resnet_seeds_arg = ",".join(str(s) for s in TRANSFER_SEEDS)
    resnet_cmd = [
        sys.executable, "-u", str(REPO_DIR / "scripts" / "run_transfer_learning.py"),
        "--model-family", PRETRAINED_RESNET_FAMILY,
        "--seeds", resnet_seeds_arg,
        "--storage-root", str(LOCAL_TRANSFER_ROOT),
    ]
    if AUTO_RESUME:
        resnet_cmd.append("--resume")
    if SKIP_COMPLETED:
        resnet_cmd.append("--skip-completed")
    if SYNC_AFTER_EPOCH:
        resnet_cmd.append("--sync-after-epoch")
    if PERSISTENT_TRANSFER_ROOT is not None:
        resnet_cmd += ["--persistent-root", str(PERSISTENT_TRANSFER_ROOT)]

    run_command(resnet_cmd, cwd=REPO_DIR, log_path=RUN_DIR / "logs" / f"{PRETRAINED_RESNET_FAMILY}_table_b.log")

    resnet_table_b_path = REPO_DIR / "results" / "transfer_learning" / "table_b.csv"
    if resnet_table_b_path.exists():
        import pandas as pd
        resnet_table_b = pd.read_csv(resnet_table_b_path)
        display(resnet_table_b)
        shutil.copy2(resnet_table_b_path, RUN_DIR / "reports" / "table_b.csv")
    else:
        print(f"{PRETRAINED_RESNET_FAMILY} run did not produce Table B -- see the log above for details.")
    sync_after_phase(PRETRAINED_RESNET_FAMILY)
else:
    print("Skipped (RUN_PRETRAINED_RESNET_EXTENSION=False).")


In [ ]:
# ============================================================
# Optional second pass: ResNet50 too, if both toggles are on (the cell
# above only runs one family per pass -- rerun it with the family swapped,
# or just duplicate the call here).
# ============================================================

if RUN_PRETRAINED_RESNET18 and RUN_PRETRAINED_RESNET50:
    PRETRAINED_RESNET_FAMILY = "pretrained_resnet50"
    resnet50_seeds_arg = ",".join(str(s) for s in TRANSFER_SEEDS)
    resnet50_cmd = [
        sys.executable, "-u", str(REPO_DIR / "scripts" / "run_transfer_learning.py"),
        "--model-family", PRETRAINED_RESNET_FAMILY,
        "--seeds", resnet50_seeds_arg,
        "--storage-root", str(LOCAL_TRANSFER_ROOT),
        "--resume", "--skip-completed", "--sync-after-epoch",
        "--persistent-root", str(PERSISTENT_TRANSFER_ROOT),
    ]
    run_command(resnet50_cmd, cwd=REPO_DIR, log_path=RUN_DIR / "logs" / f"{PRETRAINED_RESNET_FAMILY}_table_b.log")
    resnet50_table_b_path = REPO_DIR / "results" / "transfer_learning" / "table_b.csv"
    if resnet50_table_b_path.exists():
        import pandas as pd
        display(pd.read_csv(resnet50_table_b_path))
        shutil.copy2(resnet50_table_b_path, RUN_DIR / "reports" / f"{PRETRAINED_RESNET_FAMILY}_table_b.csv")
    sync_after_phase(f"{PRETRAINED_RESNET_FAMILY}_transfer_learning")
else:
    print("Skipped (needs both RUN_PRETRAINED_RESNET18=True and RUN_PRETRAINED_RESNET50=True).")


## 8. Final Drive sync

In [ ]:
# ============================================================
# Final sync -- everything under RUN_DIR (logs/reports/manifests here) plus
# whatever VOLO/pretrained-ResNet already synced to their own
# AgeGender/transfer_learning/ tree via --sync-after-epoch above.
# ============================================================

if IN_COLAB and USE_GOOGLE_DRIVE:
    drive_run_dir = Path("/content/drive/MyDrive/AgeGender/runs") / RUN_ID
    copy_tree_merge(RUN_DIR, drive_run_dir)
    print(f"Synchronized to Google Drive: {drive_run_dir}")
else:
    print("Google Drive persistence skipped (USE_GOOGLE_DRIVE=False or not running in Colab).")

print("\nDone.")
